# Классификация автоматизированного трафика Авито

**Задача**: по событиям внутри суточного окна оценить вероятность того, что `cookie_id` принадлежит трафику известных сервисов сбора данных. Основная метрика - максимальный precision при recall не ниже 70% на уровне куки.

Импортируем необходимые библиотеки. Зафиксируем константы и SEED.

In [15]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from metric import precision_at_recall

DATA_DIR = Path('data')
SEED = 42
np.random.seed(SEED)

## 1. Проверка данных и временного окна

Сначала проверяем схему, уникальность куки и состав примера ответа. Затем оставляем только события с `window_start_ts <= event_ts < window_end_ts`. Полные дубликаты удаляются после фильтрации времени; разные события в одну секунду сохраняются.

Загружаем данные

In [16]:
date_cols = ['cookie_created_at', 'window_start_ts', 'window_end_ts']
train = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=date_cols)
test = pd.read_csv(DATA_DIR / 'test.csv', parse_dates=date_cols)
events = pd.read_csv(DATA_DIR / 'events.csv', parse_dates=['event_ts'])
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

Минимальная проверка данных

In [17]:
print(f"Проверка на уникальность записей. Train: {train.cookie_id.is_unique}. Test: {test.cookie_id.is_unique}")
print(f"Куки из train и test пересекаются? - {"Нет" if set(train.cookie_id).isdisjoint(test.cookie_id) else "Да"}")
print(f"Время начала окна меньше, чем время конца в Train? - {(train.window_start_ts < train.window_end_ts).all()}")
print(f"Время начала окна меньше, чем время конца в Test? - {(test.window_start_ts < test.window_end_ts).all()}")

# Объединим train и test в один датасет
meta = pd.concat([train, test], ignore_index=True)

Проверка на уникальность записей. Train: True. Test: True
Куки из train и test пересекаются? - Нет
Время начала окна меньше, чем время конца в Train? - True
Время начала окна меньше, чем время конца в Test? - True


In [18]:
# Проверяем уникальность куки
assert meta.cookie_id.is_unique
# Проверяем, чтобы куки из events были в объединенном датасете meta
assert set(events.cookie_id).issubset(meta.cookie_id)

In [19]:
# Берем только те данные, которые входят в окно наблюдения
windowed = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id', validate='many_to_one')
windowed = windowed.loc[(windowed.event_ts >= windowed.window_start_ts) & (windowed.event_ts < windowed.window_end_ts)]

# Удаляем полные дубликаты
clean_events = windowed.drop_duplicates(subset=events.columns.tolist()).copy()

# Форматируем признак platform, так как присутствуют значения "desktop", "Web", "web" и "WEB" равны.
# То же самое и с android и ios.
clean_events['platform'] = (
    clean_events.platform.str.casefold()
    .replace({'desktop': 'web', 'iphone': 'ios'})
)

# Сортируем значения по куки и событиям.
clean_events = clean_events.sort_values(
    ['cookie_id', 'event_ts'], kind='stable'
).reset_index(drop=True)

summary = pd.Series({
    'train cookies': len(train),
    'test cookies': len(test),
    'positive train cookies': int(train.target.sum()),
    'all events': len(events),
    'events inside window': len(windowed),
    'full duplicates inside window': len(windowed) - len(clean_events),
    'clean events': len(clean_events),
})
display(summary.to_frame('count'))

,count
train cookies,11091
test cookies,4909
positive train cookies,899
all events,328905
events inside window,288126
full duplicates inside window,4293
clean events,283833


In [20]:
assert len(windowed) == 288126
assert len(clean_events) == 283833
assert clean_events.event_ts.ge(clean_events.window_start_ts).all()
assert clean_events.event_ts.lt(clean_events.window_end_ts).all()
assert clean_events.event_name.ne('captcha_shown').all()

## 2. Baseline

Воспроизводим quickstart: два признака (`n_events`, `item_nunique`) и RandomForest с 300 деревьями и `min_samples_leaf=3`. Обучение: 6-16 апреля; проверка: 17-19 апреля. Метрика берётся из `metric.py`. Прогнозы сохраняются в `artifacts/baseline/`, строка результата - в `reports/experiments.csv`.

In [21]:
# Агрегируем данные по кукам.
baseline_features = (
    windowed.groupby('cookie_id').agg(
        n_events=('event_ts', 'size'),
        item_nunique=('item_id', 'nunique'),
    ).reset_index()
)

# Создаем фрейм из двух признака: количество событий и количество уникальных объявлений.
X_baseline = (
    train[['cookie_id']]
    .merge(baseline_features, on='cookie_id', how='left', validate='one_to_one')
    .fillna({'n_events': 0, 'item_nunique': 0})
)

feature_cols = ['n_events', 'item_nunique']
split_date = pd.Timestamp('2026-04-17')
is_train = train.window_start_ts < split_date

In [22]:
# Создаем базовую модель и обучаем ее на нашем X_baseline
baseline = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=3, random_state=SEED
)
baseline.fit(X_baseline.loc[is_train, feature_cols], train.loc[is_train, 'target'])
y_valid = train.loc[~is_train, 'target']
score_valid = baseline.predict_proba(X_baseline.loc[~is_train, feature_cols])[:, 1]

In [ ]:
# Сохраняем результат в словарик. 
baseline_metrics = {
    'experiment': 'quickstart_baseline_C',
    'train_before': str(split_date.date()),
    'valid_from': str(split_date.date()),
    'n_train': int(is_train.sum()),
    'n_valid': int((~is_train).sum()),
    'n_valid_positive': int(y_valid.sum()),
    'precision_at_recall_70': precision_at_recall(y_valid, score_valid),
    'average_precision': average_precision_score(y_valid, score_valid),
    'roc_auc': roc_auc_score(y_valid, score_valid),
    'random_state': SEED,
    'deduplicate': False,
}

# Сохраняем результат baseline в ./artifacts и ставим запись в reports/experiments.csv
predictions = train.loc[~is_train, ['cookie_id', 'target']].copy()
predictions['score'] = score_valid

Path('artifacts/baseline').mkdir(parents=True, exist_ok=True)
predictions.to_csv('artifacts/baseline/validation_C.csv', index=False)
report_path = Path('reports/experiments.csv')
report_path.parent.mkdir(parents=True, exist_ok=True)

In [24]:
previous = pd.read_csv(report_path) if report_path.exists() else pd.DataFrame()
if not previous.empty:
    previous = previous.loc[previous.experiment.ne(baseline_metrics['experiment'])]
pd.concat([previous, pd.DataFrame([baseline_metrics])], ignore_index=True).to_csv(
    report_path, index=False
)
display(pd.Series(baseline_metrics).to_frame('value'))

,value
experiment,quickstart_baseline_C
train_before,2026-04-17
valid_from,2026-04-17
n_train,9140
n_valid,1951
n_valid_positive,160
precision_at_recall_70,0.106161
average_precision,0.180385
roc_auc,0.608129
random_state,42


При seed=42 результат на проверке 17–19 апреля: **P@R≥0.70 = 0.106161**, Average Precision = 0.180385, ROC-AUC = 0.608129.